In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score

DATA_PATH = r'E:\DS\tezendra\Projects\JP Morgan Job Simulation\Task 3\Task 3 and 4_Loan_Data.csv'

FEATURE_NAMES = [
    'credit_lines_outstanding',
    'loan_amt_outstanding',
    'total_debt_outstanding',
    'income',
    'years_employed',
    'fico_score'
]

df = pd.read_csv(DATA_PATH)

X = df[FEATURE_NAMES]
y = df['default']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

baseline = LogisticRegression(max_iter=1000, random_state=42)
baseline.fit(X_train_scaled, y_train)

baseline_auc = roc_auc_score(y_test, baseline.predict_proba(X_test_scaled)[:, 1])
print(f'Baseline Logistic Regression Test AUC: {baseline_auc:.4f}')

xgb = XGBClassifier(
    eval_metric='logloss',
    random_state=42
)

param_grid = {
    'n_estimators': [100, 150, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9]
}

random_search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_grid,
    n_iter=20,
    scoring='roc_auc',
    cv=5,
    n_jobs=-1,
    random_state=42
)

random_search.fit(X_train, y_train)
best_model = random_search.best_estimator_

print(f'Best AUC (CV): {random_search.best_score_:.4f}')
print(f'Best params: {random_search.best_params_}')

test_auc = roc_auc_score(y_test, best_model.predict_proba(X_test)[:, 1])
print(f'XGBoost Test AUC: {test_auc:.4f}')

importances = pd.Series(
    best_model.feature_importances_, index=FEATURE_NAMES
).sort_values(ascending=False)

print('\nFeature importances:')
print(importances.to_string())

top_feature, top_importance = importances.index[0], importances.iloc[0]
if top_importance > 0.9:
    print(f'\n"{top_feature}" accounts for {top_importance:.1%} of importance.')


def predict_expected_loss(features, model, recovery_rate=0.10):
    if isinstance(features, dict):
        X_new = pd.DataFrame([features])[FEATURE_NAMES]
    else:
        X_new = pd.DataFrame([features], columns=FEATURE_NAMES)

    pd_prob = model.predict_proba(X_new)[0, 1]
    loan_amt = X_new['loan_amt_outstanding'].iloc[0]
    expected_loss = pd_prob * loan_amt * (1 - recovery_rate)

    return {
        "Probability of Default": pd_prob,
        "Expected Loss": expected_loss
    }


if __name__ == '__main__':
    sample = X_test.iloc[0].to_dict()
    result = predict_expected_loss(sample, best_model)
    print('\nSample prediction:', result)

Baseline Logistic Regression Test AUC: 1.0000
Best AUC (CV): 0.9999
Best params: {'subsample': 0.8, 'n_estimators': 200, 'max_depth': 5, 'learning_rate': 0.1, 'colsample_bytree': 0.9}
XGBoost Test AUC: 0.9999

Feature importances:
credit_lines_outstanding    0.774561
total_debt_outstanding      0.122039
years_employed              0.054411
income                      0.028887
fico_score                  0.015575
loan_amt_outstanding        0.004527

Sample prediction: {'Probability of Default': np.float32(1.1795985e-06), 'Expected Loss': np.float64(0.003982905998809259)}
